# LLM-Assisted Topic Modeling for AI in Education

**Portfolio note:** The raw dataset is not included in this repository. Set `OPENAI_API_KEY` as an environment variable before running API cells. Do not commit API keys or raw social media data to GitHub.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%pip install -q --upgrade openai pandas xlsxwriter

In [ ]:
!pip install gensim

In [ ]:
import os, json, time, random, pandas as pd
from openai import OpenAI

In [ ]:
def chat_json(messages, model):
    try:
        r = client.chat.completions.create(
            model=model,
            messages=messages,
            response_format={"type":"json_object"},
            temperature=0
        )
        return json.loads(r.choices[0].message.content)
    except Exception:
        r = client.chat.completions.create(model=model, messages=messages, temperature=1)
        text = r.choices[0].message.content.strip()
        s, e = text.find("{"), text.rfind("}")
        return json.loads(text[s:e+1] if s!=-1 and e!=-1 else "{}")

In [ ]:
# Do not hard-code API keys in notebooks. Set OPENAI_API_KEY in your environment.
import os
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL_DISCOVERY = "gpt-5"
MODEL_CLASSIFY  = "gpt-5"


In [ ]:
df = pd.read_excel("/kaggle/input/data-2000-dosmud/cleaning data.xlsx")
texts = df["comment_clean"].dropna().astype(str).tolist()

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

try:
    stopwords.words('indonesian')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')

stop_words = set(stopwords.words('indonesian'))

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

try:
    stopwords.words('indonesian')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')

stop_words = set(stopwords.words('indonesian'))

def preprocess_for_evaluation(text):
    tokens = word_tokenize(text.lower())
    return [token for token in tokens if token.isalpha() and token not in stop_words]

tokenized_texts = [preprocess_for_evaluation(comment_clean) for comment_clean in texts]
preprocessed_for_prompt = [" ".join(doc) for doc in tokenized_texts]

In [ ]:
def discover_topics(texts, nmin=3, nmax=3):
   # sampel = random.sample(texts, k=min(sample_size, len(texts)))
    system = (
        "Anda adalah seorang analis data ahli dalam topic modeling. Tugas Anda adalah menganalisis sampel komentar "
        "dan mengekstrak topik-topik dengan kualitas statistik tertinggi."
        f" Anda harus mengusulkan {nmin}-{nmax} topik global."
        "Ikuti panduan ini untuk hasil terbaik:"
        "1.  **Prioritaskan Koherensi**: Untuk setiap topik, pilih 10 keyword yang sangat berhubungan erat dan sering muncul bersamaan dalam satu komentar."
        "2.  **Jaga Keterbacaan Data**: Semua keyword HARUS dipilih dari kata-kata yang ada di dalam teks komentar yang diberikan. Teks ini sudah dibersihkan (lowercase, tanpa stopwords dan tanda baca). JANGAN mengubah bentuk kata, mencari sinonim, atau menghasilkan kata yang tidak ada dalam daftar."
        "3.  **Usahakan Topik Berbeda**: Usahakan agar topik-topik yang dihasilkan berbeda satu sama lain, namun jangan memaksakan keunikan jika itu membuat topiknya menjadi tidak relevan atau terlalu spesifik."
        "Untuk setiap topik, berikan: id, label, deskripsi, dan 10 keyword tersebut."
        'Balas HANYA dengan format JSON: {"topics":[{"id":1,"label":"...","deskripsi":"...","keywords":["k1","k2","k3","k4","k5", "k6", "k7","k8","k9", "k10"]}, ...]}'
    )
    user = "Sampel komentar:\n\n" + "\n---\n".join(texts)
    obj = chat_json(
        [{"role":"system","content":system},{"role":"user","content":user}],
        model=MODEL_DISCOVERY
    )
    topics = obj.get("topics", [])
    out, next_id, seen = [], 1, set()
    for t in topics:
        tid = int(t.get("id", next_id))
        if tid in seen: tid = next_id
        seen.add(tid); next_id = tid+1
        out.append({
            "id": tid,
            "label": (t.get("label") or "").strip(),
            "deskripsi": (t.get("deskripsi") or "").strip(),
            "keywords": [(w or "").strip() for w in t.get("keywords", [])][:10]
        })
    if not out:
        out = []
    return out

In [ ]:
import pandas as pd
import time
from tqdm.auto import tqdm # Library untuk progress bar

all_topics_list = []
num_runs = 10 # Jumlah perulangan

print(f"Memulai {num_runs} perulangan untuk mengumpulkan topik...")

# Melakukan perulangan sebanyak num_runs
for i in tqdm(range(num_runs), desc="Generating Topics"):
    
    # Memanggil fungsi untuk menghasilkan 5 topik
    topic_set = discover_topics(
        preprocessed_for_prompt, 
        nmin=3, 
        nmax=3
    )
    
    # Menambahkan hasil dari perulangan saat ini ke dalam list utama
    for topic in topic_set:
        # Buat dictionary baru yang menyertakan nomor perulangan
        topic_data = {
            'run_number': i + 1,
            'topic_id_in_run': topic['id'],
            'label': topic['label'],
            'description': topic['deskripsi'],
            'keywords': topic['keywords']
        }
        all_topics_list.append(topic_data)
    
    print(f"Run #{i+1} selesai, menambahkan {len(topic_set)} topik ke dalam daftar.")
    
    # Beri jeda singkat antar request API
    time.sleep(2)

# Buat DataFrame final dari list yang sudah kita kumpulkan
consolidated_topics_df = pd.DataFrame(all_topics_list)

print(f"\nProses selesai. Berhasil mengumpulkan total {len(consolidated_topics_df)} topik.")

# Tampilkan DataFrame hasil gabungan
print("\n--- DataFrame Gabungan Hasil dari 10 Perulangan ---")
display(consolidated_topics_df)

In [ ]:
# Simpan DataFrame hasil gabungan ke Excel
output_file = "iterasi_topic.xlsx"
consolidated_topics_df.to_excel(output_file, index=False)

print(f"\nDataFrame hasil gabungan sudah disimpan ke file: {output_file}")

In [ ]:
import pandas as pd
import numpy as np

# ambil label dan identitas topik
topic_labels = [f"Run{row.run_number}_T{row.topic_id_in_run}" for _, row in consolidated_topics_df.iterrows()]
topic_texts  = consolidated_topics_df["label"].astype(str).tolist()

# ubah label jadi set kata
topic_sets = [set(txt.split()) for txt in topic_texts]

# buat matriks jaccard
n = len(topic_sets)
jaccard_mat = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        inter = len(topic_sets[i].intersection(topic_sets[j]))
        union = len(topic_sets[i].union(topic_sets[j]))
        jaccard_mat[i, j] = inter / union if union > 0 else 0

# bungkus ke DataFrame
jaccard_df = pd.DataFrame(jaccard_mat, index=topic_labels, columns=topic_labels)
print("\n=== Matriks Jaccard Similarity Antar Topik (berdasarkan label) ===")
display(jaccard_df.round(3))

In [ ]:
# Simpan DataFrame hasil gabungan ke Excel
output_file = "jaccard.xlsx"
jaccard_df.to_excel(output_file)

print(f"\nDataFrame hasil gabungan sudah disimpan ke file: {output_file}")

In [ ]:
import pandas as pd

# Asumsikan jaccard_df sudah ada
topic_index = jaccard_df.index

# Ambil daftar run unik
runs = sorted(set(lbl.split("_")[0] for lbl in topic_index))

result_rows = []

for topic in topic_index:
    run_origin = topic.split("_")[0]
    row_result = {"Topik": topic}
    
    for run in runs:
        if run == run_origin:
            continue  # skip perbandingan dengan run yang sama
        
        # filter kolom untuk run tertentu
        candidates = jaccard_df.loc[topic, jaccard_df.columns.str.startswith(run)]
        
        # ambil topik dengan skor tertinggi
        best_match = candidates.idxmax()
        best_score = candidates.max()
        
        row_result[f"{run}_best"] = best_match
        row_result[f"{run}_score"] = round(best_score, 3)
    
    result_rows.append(row_result)

# Bungkus ke DataFrame
best_per_run_df = pd.DataFrame(result_rows)

print("\n=== Tabel Topik vs Topik Paling Mirip di Setiap Run ===")
display(best_per_run_df)

In [ ]:
# Simpan ke Excel
best_per_run_df.to_excel("best_match_per_run.xlsx", index=False)
print("Hasil disimpan ke: best_match_per_run.xlsx")

In [ ]:
iterasi_df = consolidated_topics_df
iterasi_df

In [ ]:
best_df = best_per_run_df
best_df

In [ ]:
# Buat ID unik: RunX_TY
iterasi_df["Topik"] = iterasi_df.apply(
    lambda r: f"Run{r['run_number']}_T{r['topic_id_in_run']}", axis=1
)

In [ ]:
# 2. Siapkan hasil gabungan
merged_rows = []

for _, row in best_df.iterrows():
    origin_topic = row["Topik"]
    
    # ambil semua kolom *_best
    related_topics = [val for col, val in row.items() if col.endswith("_best") and pd.notna(val)]
    all_topics = [origin_topic] + related_topics  # termasuk topik asal
    
    # kumpulkan keywords (tanpa buang duplikat)
    kw_list = []
    for t in all_topics:
        kws = iterasi_df.loc[iterasi_df["Topik"] == t, "keywords"].values
        if len(kws) > 0:
            if isinstance(kws[0], str):
                kw_list.extend(kws[0].split(", "))  # asumsikan dipisah koma
            else:
                kw_list.extend(kws[0])
    
    merged_keywords = ", ".join(kw_list)  # tidak buang duplikat
    
    merged_rows.append({
        "Topik_Asal": origin_topic,
        "Related_Topics": ", ".join(all_topics),
        "Merged_Keywords": merged_keywords
    })

In [ ]:
# 3. Hasil akhir
merged_df = pd.DataFrame(merged_rows)

print("\n=== Tabel Gabungan Keywords (dengan duplikat) ===")
display(merged_df)

In [ ]:
# 4. Simpan ke Excel
merged_df.to_excel("merged_keywords.xlsx", index=False)
print("Hasil disimpan ke: merged_keywords.xlsx")

In [ ]:
import pandas as pd
from collections import Counter
import ast

# Fungsi untuk mengubah Merged_Keywords -> list keyword bersih
def to_keyword_list(x):
    if pd.isna(x):
        return []

    # kalau masih string bentuk "['ai', 'chatgpt', ...]"
    if isinstance(x, str):
        try:
            obj = ast.literal_eval(x)   # parse jadi list Python
        except Exception:
            # fallback: split manual
            tokens = x.split(",")
            return [t.strip(" []'\"") for t in tokens]
    else:
        obj = x

    # flatten kalau list di dalam list
    if isinstance(obj, (list, tuple, set)):
        out = []
        for item in obj:
            if isinstance(item, (list, tuple, set)):
                out.extend(item)
            else:
                out.append(item)
    else:
        out = [obj]

    # bersihkan sisa bracket/tanda petik
    clean = [str(t).strip(" []'\"") for t in out if str(t).strip(" []'\"") != ""]
    return clean

# Fungsi untuk menghitung frekuensi keyword per baris
def count_keywords_per_row(x):
    kw_list = to_keyword_list(x)
    c = Counter(kw_list)
    # balikin dalam bentuk dict: {'tugas': 9, 'skripsi': 8, ...}
    return dict(c.most_common())

# Kolom baru: Keyword_Frequencies
merged_df["Keyword_Frequencies"] = merged_df["Merged_Keywords"].apply(count_keywords_per_row)
merged_df

In [ ]:
# 4. Simpan ke Excel
merged_df.to_excel("merged_keywords_with_frequencies.xlsx", index=False)
print("Hasil disimpan ke: merged_keywords_with_frequencies.xlsx")

In [ ]:
# Opsi A: pakai jumlah topik di iterasi pertama (Run 1)
n = iterasi_df.loc[iterasi_df["run_number"] == 1, "topic_id_in_run"].nunique()
print("n (topik per iterasi, dari Run1) =", n)

merged_df = merged_df.head(int(n))
merged_df

In [ ]:
# 2. Pastikan kolom Keyword_Frequencies dibaca sebagai list of tuple (kalau string, eval)
def parse_freq_list(x):
    if isinstance(x, str):
        try:
            return eval(x)  # contoh: "[('ai', 10), ('guru', 10)]"
        except:
            return []
    return x

merged_df.loc[:, "Keyword_Frequencies"] = merged_df["Keyword_Frequencies"].apply(parse_freq_list)
merged_df

In [ ]:
from collections import Counter

def get_top10_keywords(freqs):
    if freqs is None:
        return ""

    # jika dict: {'ai': 10, 'guru': 8, ...}
    if isinstance(freqs, dict):
        items = list(freqs.items())
    # jika list of tuples: [('ai', 10), ('guru', 8), ...]
    elif isinstance(freqs, list):
        try:
            items = [(k, v) for k, v in freqs]
        except Exception:
            return ""
    else:
        return ""

    # urutkan: frekuensi menurun, lalu keyword naik (biar konsisten)
    items_sorted = sorted(items, key=lambda x: (-x[1], str(x[0])))

    # ambil 10 teratas
    top10 = items_sorted[:10]

    # format jadi string: 'ai' (10), 'guru' (8), ...
    return ", ".join([f"'{k}' ({v})" for k, v in top10])

# buat kolom baru Top10_Keywords
merged_df["Top10_Keywords"] = merged_df["Keyword_Frequencies"].apply(get_top10_keywords)
merged_df

In [ ]:
import pandas as pd
from collections import Counter

# Pastikan iterasi_df sudah ada
iterasi_df["Topik"] = iterasi_df.apply(
    lambda r: f"Run{r['run_number']}_T{r['topic_id_in_run']}", axis=1
)

# Buat mapping Topik → Label
topic_to_label = dict(zip(iterasi_df["Topik"], iterasi_df["label"]))

# Tempat menyimpan hasil
dominant_labels = []
label_counts_all = []

for _, row in merged_df.iterrows():
    related = str(row["Related_Topics"]).split(", ")
    # Ambil semua label yang terkait
    labels_found = [topic_to_label.get(t, None) for t in related if t in topic_to_label]
    labels_found = [l for l in labels_found if l]  # hapus None
    
    if labels_found:
        counter = Counter(labels_found)
        # Simpan distribusi label dan jumlahnya
        label_counts_all.append(dict(counter))
        # Ambil label yang paling sering
        dominant_label = counter.most_common(1)[0][0]
    else:
        label_counts_all.append({})
        dominant_label = "-"
    
    dominant_labels.append(dominant_label)

# Tambahkan ke merged_df
merged_df["Label_Distribution"] = label_counts_all
merged_df["Dominant_Label"] = dominant_labels

# Tampilkan hasil
print("\n=== Contoh hasil label dan frekuensi tiap baris ===")
merged_df[["Topik_Asal", "Label_Distribution", "Dominant_Label"]]

In [ ]:
# 5. Pilih hanya kolom yang diminta
output_df = merged_df[["Topik_Asal", "Related_Topics", "Merged_Keywords", "Top10_Keywords", "Dominant_Label"]]
output_df

In [ ]:
import re

# Bersihkan angka dalam kurung dari kolom Top10_Keywords
output_df["Top10_Keywords_Clean"] = (
    output_df["Top10_Keywords"]
    .astype(str)
    .apply(lambda x: re.sub(r"\s*\(\d+\)", "", x))   # hapus pola " (angka)"
)

# Pilih kolom yang diminta, pakai versi bersih
output_df = output_df[["Topik_Asal", "Related_Topics", "Merged_Keywords", "Top10_Keywords_Clean", "Dominant_Label"]]
output_df = output_df.rename(columns={"Top10_Keywords_Clean": "Top10_Keywords"})
output_df = output_df.rename(columns={"Dominant_Label":"Label"})

print("\n=== Hasil Top10_Keywords tanpa angka ===")
output_df

In [ ]:
# Simpan ke Excel
output_df.to_excel("final_top20_keywords_clean.xlsx", index=False)
print("Hasil disimpan ke: final_top20_keywords_clean.xlsx")

In [ ]:
df_evaluasi = pd.read_excel("/kaggle/input/data-2000-dosmud/cleaning data.xlsx")

In [ ]:
import re
import pandas as pd
import collections
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# =========================
# 1) Pakai teks preprocessing kamu
# =========================
# df_evaluasi["final_preprocess_comment"] adalah string token yang sudah dibersihkan
texts2 = df_evaluasi["comment_clean"].dropna().astype(str).tolist()
tokenized_texts2 = [t.split() for t in texts2]
dictionary = Dictionary(tokenized_texts2)
texts = texts2

# =========================
# 2) Bangun FINAL_TOPIC_SET dari output_df (Top10_Keywords + Label)
#    + bersihkan " (10)" dll dan karakter non-alfanumerik di ujung
# =========================
def clean_top10_string(s: str):
    # hapus " (angka)" -> jadi token saja
    s = re.sub(r"\s*\(\d+\)\s*", "", str(s))
    # split, strip, dan buang token kosong
    toks = [w.strip() for w in s.split(",") if w.strip()]
    # opsional: normalisasi ringan -> ambil alfanumerik dasar
    toks = [re.sub(r"[^0-9a-zA-Z_\-]+", "", w) for w in toks]
    toks = [w for w in toks if w]  # buang yang kosong setelah cleaning
    return toks

topic_table = output_df.copy()
FINAL_TOPIC_SET = []
for i, row in topic_table.iterrows():
    kw_clean = clean_top10_string(row.get("Merged_Keywords", ""))
    if kw_clean:  # hanya ambil topik yang punya kata kunci
        FINAL_TOPIC_SET.append({
            "id": int(i) + 1,
            "label": str(row.get("Label", "")).strip() or f"Topic {i+1}",
            "deskripsi": str(row.get("Label", "")).strip(),
            "keywords": kw_clean[:10]
        })

# Safety: jika semua kosong, hentikan
if not FINAL_TOPIC_SET:
    raise ValueError("FINAL_TOPIC_SET kosong. Pastikan kolom Top10_Keywords berisi kata kunci yang valid.")

# =========================
# 3) Model topik sederhana utk evaluasi
# =========================
class LLMTopicModel:
    def __init__(self, FINAL_TOPIC_SET):
        self._topics = [t['keywords'] for t in FINAL_TOPIC_SET]
        self.num_topics = len(self._topics)

    def show_topic(self, topicid, topn=10):
        keywords = self._topics[topicid][:topn]
        return [(word, 1.0) for word in keywords]

def _get_topic_words(model, topn=10):
    return [[word for word, _ in model.show_topic(t, topn)] for t in range(model.num_topics)]

# =========================
# 4) Metrik evaluasi
# =========================
def calculate_coherence_cv(model, texts, dictionary, topn=10):
    topics = _get_topic_words(model, topn)
    # buang topik kosong jika ada
    topics = [t for t in topics if t]
    if not topics:
        return 0.0
    cm = CoherenceModel(topics=topics, texts=texts, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def calculate_topic_uniqueness(model, topn=10):
    topics = _get_topic_words(model, topn)
    topics = [t for t in topics if t]
    if not topics:
        return 0.0
    all_words = [word for topic in topics for word in topic]
    word_counts = collections.Counter(all_words)
    topic_scores = []
    for topic in topics:
        tu_k_sum = sum(1 / word_counts[word] for word in topic if word in word_counts)
        tu_k = tu_k_sum / max(len(topic), 1)
        topic_scores.append(tu_k)
    return sum(topic_scores) / len(topic_scores) if topic_scores else 0.0

def calculate_document_coverage(model, texts, topn=10):
    total_docs = len(texts)
    if total_docs == 0:
        return 0.0
    topics = _get_topic_words(model, topn)
    all_topic_words = set(word for topic in topics for word in topic)
    if not all_topic_words:
        return 0.0
    covered_docs_count = sum(1 for doc in texts if not all_topic_words.isdisjoint(doc))
    return covered_docs_count / total_docs

def calculate_factuality(model, texts, topn=10):
    corpus_vocab = set(word for doc in texts for word in doc)
    topic_vocab = set(word for topic in _get_topic_words(model, topn) for word in topic)
    if not topic_vocab:
        return 0.0
    intersection = topic_vocab.intersection(corpus_vocab)
    return len(intersection) / len(topic_vocab)

def evaluate_llm_topics(FINAL_TOPIC_SET, texts, dictionary):
    model = LLMTopicModel(FINAL_TOPIC_SET)
    cv = calculate_coherence_cv(model, texts, dictionary)
    tu = calculate_topic_uniqueness(model)
    dc = calculate_document_coverage(model, texts)
    fa = calculate_factuality(model, texts)
    return {"coherence_cv": cv, "topic_uniqueness": tu, "coverage_dc": dc, "factuality": fa}

# =========================
# 5) Jalankan evaluasi (pakai texts2!)
# =========================
evaluation_results = evaluate_llm_topics(FINAL_TOPIC_SET, tokenized_texts2, dictionary)

print("\n=== Rangkuman Evaluasi Model Topik ===")
summary_df = pd.DataFrame(list(evaluation_results.items()), columns=['Metrik', 'Skor'])
print(summary_df.to_string(index=False))

In [ ]:
import pandas as pd
import time
import ast

TEXT_COL = "comment_clean"

def _parse_keywords(x):
    if isinstance(x, list):
        return [str(w).strip() for w in x if str(w).strip()]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            val = ast.literal_eval(s)
            if isinstance(val, list):
                return [str(w).strip() for w in val if str(w).strip()]
        except Exception:
            pass
    return [w.strip() for w in s.split(",") if w.strip()]


def _build_topics_from_output_df(output_df: pd.DataFrame):
    topics = output_df.copy().reset_index(drop=True)
    topics["topic_id"] = topics.index + 1

    if "Label" not in topics.columns or "Top10_Keywords" not in topics.columns:
        raise ValueError("output_df wajib punya kolom 'Label' dan 'Top10_Keywords'.")

    topics["keywords"] = topics["Top10_Keywords"].apply(_parse_keywords)

    topic_list = []
    for _, r in topics.iterrows():
        topic_list.append({
            "id": int(r["topic_id"]),
            "label": str(r["Label"]).strip(),
            "keywords": [k.strip() for k in r["keywords"][:10]]
        })

    id_to_label = {t["id"]: t["label"] for t in topic_list}
    return topic_list, id_to_label


def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def _call_assign_batch(chat_json, model, system_prompt, topic_block, batch, text_col):
    comments_block = "\n".join([f'row_id={r["row_id"]} :: {r[text_col]}' for r in batch])

    user = (
        "Katalog topik (WAJIB pilih salah satu id):\n"
        f"{topic_block}\n\n"
        "Komentar yang harus diklasifikasikan:\n"
        f"{comments_block}"
    )

    obj = chat_json(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": user}],
        model=model
    )

    assigns = obj.get("assignments", [])
    return assigns


def assign_topics_gpt(
    df: pd.DataFrame,
    output_df: pd.DataFrame,
    text_col: str = TEXT_COL,
    batch_size: int = 25,
    sleep_sec: float = 1.0,
    model: str = MODEL_DISCOVERY,  # pastikan sudah ada
):
    topics, id_to_label = _build_topics_from_output_df(output_df)
    allowed_ids = set(t["id"] for t in topics)

    # === PROMPT UTAMA (pakai prompt kamu) ===
    system_main = (
        "Klasifikasikan setiap komentar ke tepat satu topik dari katalog.\n"
        "Pilih tepat satu topik dari katalog.\n"
        "JANGAN pilih 0 kecuali komentar benar-benar spam/promo/jauh dari AI/pendidikan.\n"
        "Komentar bisa pendek, slang, atau tidak baku; tetap tebak maksudnya.\n"
        "Jika menyebut AI/ChatGPT/guru/sekolah/ujian/belajar, anggap RELEVAN.\n"
        'Balas HANYA JSON: {"assignments":[{"row_id":..., "topic_id":..., "confidence":0.0-1.0}, ...]}\n'
    )

    # === PROMPT RETRY (kalau masih 0, paksa pilih topik terdekat) ===
    system_retry = (
        "Klasifikasikan setiap komentar ke tepat satu topik dari katalog.\n"
        "WAJIB pilih satu topik dari katalog (topic_id 1..K). 0 DILARANG.\n"
        "Pilih topik yang PALING MENDEKATI, jangan ragu.\n"
        "Komentar bisa pendek/slang; tetap tebak maksudnya.\n"
        'Balas HANYA JSON: {"assignments":[{"row_id":..., "topic_id":..., "confidence":0.0-1.0}, ...]}\n'
    )

    topic_block = "\n".join(
        [f'- topic_id={t["id"]} | label="{t["label"]}" | keywords={t["keywords"]}' for t in topics]
    )

    # pakai index asli sebagai row_id
    rows = df[[text_col]].fillna("").astype(str).reset_index().rename(columns={"index": "row_id"})
    records = rows.to_dict("records")

    all_assignments = []

    # =========================
    # PASS 1: assign normal
    # =========================
    for batch in _chunks(records, batch_size):
        assigns = _call_assign_batch(chat_json, model, system_main, topic_block, batch, text_col)

        # mapping cepat untuk cek coverage
        got = {int(a.get("row_id")): a for a in assigns if a.get("row_id") is not None}

        # pastikan semua row_id di batch ada hasilnya
        for r in batch:
            rid = int(r["row_id"])
            a = got.get(rid, {"row_id": rid, "topic_id": 0, "confidence": 0.0})

            tid = int(a.get("topic_id", 0))
            conf = float(a.get("confidence", 0.0))

            # validasi id
            if tid != 0 and tid not in allowed_ids:
                tid = 0
                conf = 0.0

            all_assignments.append({"row_id": rid, "topic_id": tid, "confidence": conf})

        if sleep_sec:
            time.sleep(sleep_sec)

    assign_df = pd.DataFrame(all_assignments).drop_duplicates(subset=["row_id"], keep="last")

    # =========================
    # PASS 2: retry untuk yang 0 (paksa tidak boleh 0)
    # =========================
    need_retry = assign_df[assign_df["topic_id"].eq(0)]["row_id"].tolist()

    if len(need_retry) > 0:
        retry_records = [r for r in records if int(r["row_id"]) in set(need_retry)]

        retry_assignments = []
        for batch in _chunks(retry_records, batch_size):
            assigns = _call_assign_batch(chat_json, model, system_retry, topic_block, batch, text_col)

            got = {int(a.get("row_id")): a for a in assigns if a.get("row_id") is not None}

            for r in batch:
                rid = int(r["row_id"])
                a = got.get(rid, {"row_id": rid, "topic_id": 1, "confidence": 0.3})

                tid = int(a.get("topic_id", 1))
                conf = float(a.get("confidence", 0.3))

                # validasi id + paksa bukan 0
                if tid == 0 or tid not in allowed_ids:
                    tid = 1  # fallback paling aman (topik pertama)
                    conf = min(conf, 0.3)

                retry_assignments.append({"row_id": rid, "topic_id": tid, "confidence": conf})

            if sleep_sec:
                time.sleep(sleep_sec)

        retry_df = pd.DataFrame(retry_assignments).drop_duplicates(subset=["row_id"], keep="last")

        # update hasil pass 1
        assign_df = assign_df.merge(retry_df, on="row_id", how="left", suffixes=("", "_retry"))
        assign_df["topic_id"] = assign_df["topic_id_retry"].fillna(assign_df["topic_id"]).astype(int)
        assign_df["confidence"] = assign_df["confidence_retry"].fillna(assign_df["confidence"]).astype(float)
        assign_df = assign_df[["row_id", "topic_id", "confidence"]]

    # =========================
    # MERGE balik ke df
    # =========================
    out = df.reset_index().rename(columns={"index": "row_id"}).merge(assign_df, on="row_id", how="left")
    out["topic_id"] = out["topic_id"].fillna(0).astype(int)
    out["confidence"] = out["confidence"].fillna(0.0).astype(float)
    out["topic_label"] = out["topic_id"].map(id_to_label).fillna("lainnya")
    out = out.set_index("row_id").sort_index()
    return out

In [ ]:
# =========================
# RUN
# =========================
df_assigned = assign_topics_gpt(
    df=df,
    output_df=output_df,
    text_col="comment_clean",
    batch_size=25,
    sleep_sec=1.0,
    model=MODEL_DISCOVERY
)

df_assigned[["comment_clean", "topic_id", "topic_label", "confidence"]]

In [ ]:
# Simpan ke Excel
df_assigned.to_excel("assign_datacleaning.xlsx", index=False)
print("Hasil disimpan ke: assign_datacleaning.xlsx")